AtlasNet baseline. Runtime: L4. Clone `f3dr_dev`, unzip the Drive zip into `data/f3d`, then `cache` / `train` / `predict`.

In [ ]:
import os
from pathlib import Path

repo = Path("/content/fve_bosch")
if not (repo / "baselines" / "atlas.py").is_file():
    !git clone -b f3dr_dev --depth 1 https://github.com/TinhAnhGitHub/fve_bosch.git {repo}
os.chdir(repo)
print(repo)

In [ ]:
import glob, os, zipfile
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

src = glob.glob("/content/drive/MyDrive/0a8f511e*.zip")[0]
size = os.path.getsize(src)
print(src, size, flush=True)
dst = Path("/content/data.zip")
done = dst.stat().st_size if dst.exists() else 0
with open(src, "rb") as f, open(dst, "ab" if done else "wb") as g:
    if done:
        f.seek(done)
    while done < size:
        buf = f.read(64 * 1024 * 1024)
        if not buf:
            break
        g.write(buf)
        done += len(buf)
        print(done, flush=True)
print("copied", done)

dest = Path("/content/fve_bosch/data/f3d")
dest.mkdir(parents=True, exist_ok=True)
if not (dest / "train").is_dir():
    print("unzip", flush=True)
    with zipfile.ZipFile(dst) as z:
        z.extractall(dest)
print("train", len(list((dest / "train").rglob("rgb_industrial.png"))))
print("val", len(list((dest / "val").rglob("rgb_industrial.png"))))

In [ ]:
!pip -q install trimesh
import torch
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"free {free/1e9:.1f} / {total/1e9:.1f} GB")

In [ ]:
!python baselines/atlas.py cache

In [ ]:
!python baselines/atlas.py train

In [ ]:
!python baselines/atlas.py predict

In [ ]:
from google.colab import files
files.download("submissions/atlas/preds_geometry.zip")